In [11]:
%%writefile config_classifiers.py
# Number of folds for cross-validation
N_FOLDS = 5

# Scoring method for hyperparameter optimization
# Options: 'AUC', 'KAPPA', 'accuracy', 'f1_macro'
CV_SCORER = 'AUC'

# =============================================================================
# ACOUSTIC FEATURE EXTRACTION CONFIGURATION
# =============================================================================

# OpenSmile feature sets to extract
# Options: 'eGeMAPSv02', 'ComParE_2016'
ACOUSTIC_FEATURE_SETS = ['eGeMAPSv02', 'ComParE_2016']

# Questions to process for acoustic features
# Options: 'Q4', 'Q6', 'Q10', 'Q12', 'ALL'
ACOUSTIC_QUESTIONS = ['Q4', 'Q6', 'Q10', 'Q12', 'ALL']

# Audio sampling rate (None = keep original)
AUDIO_SAMPLE_RATE = None

# =============================================================================
# LINGUISTIC FEATURE EXTRACTION CONFIGURATION
# =============================================================================

# Transformer models to use for linguistic analysis
# Options: 'BART', 'DistilBERT', 'RoBERTa'
LINGUISTIC_MODELS = ['BART', 'DistilBERT', 'RoBERTa']

# Questions to process for linguistic features
# Options: 'Q4', 'Q6', 'Q10', 'Q12', 'ALL'
LINGUISTIC_QUESTIONS = ['Q4', 'Q6', 'Q10', 'Q12', 'ALL']

# Maximum sequence length for tokenization
MAX_SEQUENCE_LENGTH = 512

# Training parameters for transformer models
TRANSFORMER_EPOCHS = 30
TRANSFORMER_BATCH_SIZE = 64
TRANSFORMER_LEARNING_RATES = [0.0001, 0.00005, 0.00001, 0.000005]

# Number of classes for classification
NUM_CLASSES = 2

# =============================================================================
# CLASSIFICATION CONFIGURATION
# =============================================================================

# Classification types to perform
# Options: '2-way', '3-way'
CLASSIFICATION_TYPES = ['2-way']

# Hyperparameter optimization approaches
# Options: 'simple', 'grid'
HYPERPARAMETER_OPTIMIZATION = ['simple']

# Classifiers to use (indices correspond to classifier types)
# 1: Logistic Regression, 2: KNN, 3: SVM, 4: MLP, 5: MLP_TF
CLASSIFIER_INDICES = [1, 3]  # LR and SVM by default

# =============================================================================
# HYPERPARAMETER GRIDS
# =============================================================================

# Logistic Regression hyperparameters
LR_C_VALUES = 10  # Number of C values in logspace(-5, 5)
LR_L1_RATIOS = 6  # Number of L1 ratio values in linspace(0, 1)
LR_MAX_ITER = 100000000
LR_SOLVER = 'saga'
LR_PENALTY = 'elasticnet'

# SVM hyperparameters
SVM_C_VALUES = 30  # Number of C values in logspace(-7, 7)
SVM_GAMMA_VALUES = 20  # Number of gamma values in logspace(-5, 5)
SVM_RANDOM_STATES = [1, 10, 20]

# MLP hyperparameters
MLP_ALPHA_VALUES = 5  # Number of alpha values in logspace(-7, 5)
MLP_MOMENTUM_VALUES = 5  # Number of momentum values in linspace(0, 1)
MLP_RANDOM_STATES = 5  # Number of random states in linspace(1, 10)
MLP_MAX_ITER = 10000000

# KNN hyperparameters
KNN_N_NEIGHBORS = [50, 200, 500, 2000]
KNN_LEAF_SIZES = [2, 5, 10, 30]
KNN_P_VALUES = [1, 2]
KNN_WEIGHTS = ['uniform', 'distance']

# =============================================================================
# DATA PREPROCESSING CONFIGURATION
# =============================================================================

# Feature scaling method
# Options: 'robust', 'standard', 'minmax', None
FEATURE_SCALING = 'robust'

# Handle missing values
HANDLE_MISSING_VALUES = True
MISSING_VALUE_STRATEGY = 'mean'  # Options: 'mean', 'median', 'drop'

# =============================================================================
# EVALUATION CONFIGURATION
# =============================================================================

# Metrics to calculate
EVALUATION_METRICS = ['f1_macro', 'precision_macro', 'recall_macro', 'accuracy']

# Use majority voting for final predictions
USE_MAJORITY_VOTING = True

# Threshold for binary classification (when using majority voting)
BINARY_THRESHOLD = 0.5

# Threshold for 3-way classification (when using majority voting)
TERNARY_THRESHOLD = 0.333333

# Verbose output level
# 0: Minimal output, 1: Standard output, 2: Detailed output
VERBOSE_LEVEL = 1

# =============================================================================
# FILE PATHS AND DIRECTORIES
# =============================================================================

# Base directory (relative to script location)
BASE_DIR = '..'

# Data subdirectories
DATA_SUBDIR = '/data/CognoSpeak_results/data/'
FEATURES_SUBDIR = '/data/CognoSpeak_results/feats/'
RESULTS_SUBDIR = '/data/CognoSpeak_results/results/'

# Metadata file
METADATA_FILE = '/data/CognoSpeak_results/metadata.csv'

# =============================================================================
# PARALLEL PROCESSING CONFIGURATION
# =============================================================================

# Default number of parallel jobs (will be overridden by command line argument)
DEFAULT_N_JOBS = 1

# =============================================================================
# MODEL SAVING CONFIGURATION
# =============================================================================

# Whether to save trained models
SAVE_MODELS = False

# Model file format
MODEL_FILE_EXTENSION = '.save'

# =============================================================================
# RANDOM SEED CONFIGURATION
# =============================================================================

# Random seed for reproducibility
RANDOM_SEED = 42

# =============================================================================
# CUDA CONFIGURATION (for linguistic models)
# =============================================================================

# Default CUDA device IDs
DEFAULT_CUDA_DEVICES = [0]

# =============================================================================
# FEATURE EXTRACTION FLAGS
# =============================================================================


Overwriting config_classifiers.py


In [1]:
import os,  sys, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
import spacy
%pip install torch
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from scipy.signal import butter, sosfilt
from pathlib import Path
import soundfile as sf

# Download the model
!python -m spacy download en_core_web_lg

%pip install opensmile
import opensmile

'''from transformers import (
    BertTokenizer, BertModel, get_linear_schedule_with_warmup,
    RobertaTokenizer, RobertaForSequenceClassification,
    DistilBertTokenizer, DistilBertForSequenceClassification,
    BartTokenizer, BartForConditionalGeneration,  # Note: BartForSequenceClassification doesn't exist
    AutoTokenizer, AutoModelForSequenceClassification  # More flexible alternative
)'''

from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
    precision_recall_fscore_support as score
)
from sklearn.utils.class_weight import compute_class_weight

%pip install librosa
import librosa
import config_classifiers

Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 2.9 MB/s eta 0:00:0000:0100:04
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
def load_metadata(base_dir):
    # Paths
    data_path = os.path.join(base_dir, 'audio_files/')
    feat_path = os.path.join(base_dir, 'feats/')
    results_path = os.path.join(base_dir, 'results/')

    # Load metadata
    metadata_file = os.path.join(base_dir, 'data/metadata.csv')
    df_meta = pd.read_csv(metadata_file)
    # Feature file
    df_feat_file = os.path.join(feat_path, 'CognoSpeak_eGeMAPSv02.csv')
    
    if os.path.exists(df_feat_file):
        df_feat = pd.read_csv(df_feat_file)
    else:

        df_feat = generate_features(df_meta, data_path)
        df_feat.to_csv(df_feat_file, index=False)
    
    return df_meta, df_feat, data_path, feat_path, results_path

def generate_features(df_meta, data_path):
    df_feat = pd.DataFrame(columns=['participant_id', 'Q_type', 'audio_path', 'duration_sec'])

    for participant in tqdm(df_meta['participant_id']):
        l_q_types = ['Q4']
        participant_folder = os.path.join(data_path, participant)

        participant_mp3_files = []
        
        for q_type in l_q_types:
            mp3_pattern = os.path.join(participant_folder, f"{participant}_{q_type}.mp3")
            found_files = glob.glob(mp3_pattern)
            participant_mp3_files.extend(found_files)
        
        print(f"Total files found: {len(participant_mp3_files)}")
            
        for mp3_file in participant_mp3_files:
            q_type = mp3_file.split('_')[-1].replace('.mp3', '')
            
            y, sr = librosa.load(mp3_file, sr=None)
            duration = len(y) / sr
            df_feat.loc[len(df_feat)] = [participant, q_type, mp3_file, duration]
 
    df_feat = df_feat.merge(df_meta, on='participant_id', how='inner')
    return df_feat

base_dir = '/Users/minhphan/Documents/DementiaSpeak/generated-data'
df_meta, df_feat, data_path, feat_path, results_path = load_metadata(base_dir)

print(df_meta)
print(df_feat)

    participant_id  age gender diagnosis      ethnicity  labels FOLD_0 FOLD_1  \
0  participant_001   62      F        HC  White British       0  TRAIN   TEST   
1  participant_002   58      M        HC  White British       0  TRAIN  TRAIN   
2  participant_003   69      F        HC          Asian       0   TEST  TRAIN   
3  participant_004   64      M        HC    Other White       0  TRAIN  TRAIN   
4  participant_005   71      F        HC  White British       0  TRAIN  TRAIN   
5  participant_006   78      M  Dementia  White British       1  TRAIN   TEST   
6  participant_007   82      F  Dementia  White British       1  TRAIN  TRAIN   
7  participant_008   75      M  Dementia          Asian       1   TEST  TRAIN   
8  participant_009   80      F  Dementia          Mixed       1  TRAIN  TRAIN   
9  participant_010   77      M  Dementia  White British       1  TRAIN  TRAIN   

  FOLD_2 FOLD_3 FOLD_4  
0  TRAIN  TRAIN  TRAIN  
1   TEST  TRAIN  TRAIN  
2  TRAIN  TRAIN  TRAIN  
3  TRAIN

In [3]:
def bandpass_filter(audio, sr, lowcut=200, highcut=3400, order=5):
    sos = butter(order, [lowcut, highcut], btype='band', fs=sr, output='sos')
    return sosfilt(sos, audio)

def load_sound(dir):
    audio_base_path = os.path.join(dir, "generated-data", "audio_files")
    sub = np.sort(glob.glob(os.path.join(audio_base_path, 'participant_*')))
    mp3_files = []  
    
    for s in sub:
        mp3 = glob.glob(os.path.join(s, '*Q4.mp3'))
        mp3_files.extend(mp3)


    audio_dict = {}

    for mp3_file in mp3_files:
        y, sr = librosa.load(mp3_file, sr=None)
        audio_dict[mp3_file] = {'audio': y, 'sr': sr, 'duration': len(y)/sr}
        
    return audio_dict

In [4]:
import scipy.signal as ss

In [5]:
def spectral_analysis(audio, sr):
    f, p = ss.welch(audio, fs = sr, nperseg = 256, noverlap = 200 )
    
    return plt.plot(f, p)


In [71]:
def normalize_loudness(audio, sr):
    
    # RMS normalization
    rms = np.sqrt(np.mean(audio**2))
    if rms > 0:
        audio = audio * (0.1 / rms)
    return audio

def denoise_audio(audio, sr):
    
    # Spectral subtraction denoising
    noise_sample = audio[:int(0.5 * sr)]  # First 0.5s as noise
    
    stft_audio = librosa.stft(audio)
    stft_noise = librosa.stft(noise_sample)
    
    noise_profile = np.mean(np.abs(stft_noise), axis=1, keepdims=True)
    denoised_stft = np.maximum(np.abs(stft_audio) - noise_profile, 0) * np.exp(1j * np.angle(stft_audio))
    
    return librosa.istft(denoised_stft)


def get_vad_segments(audio, sr, threshold=0.5):

    model, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad', model='silero_vad', force_reload=False)
    (get_speech_timestamps, _, _, _, _) = utils
    
    audio_tensor = torch.FloatTensor(audio)
    speech_timestamps = get_speech_timestamps(audio_tensor, model, sampling_rate=sr, threshold=threshold)
    
    return speech_timestamps

def compute_vad_stats(speech_timestamps, total_duration, sr):
    # VAD statistics: speech ratio, pause count, average pause duration
    speech_duration = sum([(seg['end'] - seg['start']) / sr for seg in speech_timestamps])
    speech_ratio = speech_duration / total_duration
    
    pause_count = len(speech_timestamps) - 1
    pause_durations = [(speech_timestamps[i+1]['start'] - speech_timestamps[i]['end']) / sr 
                      for i in range(len(speech_timestamps) - 1)]
    avg_pause_duration = np.mean(pause_durations) if pause_durations else 0
    
    return {
        'speech_ratio': speech_ratio,
        'pause_count': pause_count,
        'avg_pause_duration': avg_pause_duration,
        'total_duration': total_duration,
        'speech_duration': speech_duration
    }
    
def amplify_audio(y, sr, target_dB=-20.0):
    """Amplify audio to target dB level"""
    
    current_dB = 20 * np.log10(np.sqrt(np.mean(y**2)))
    gain_dB = target_dB - current_dB
    gain_linear = 10 ** (gain_dB / 20)
    
    return y * gain_linear


def preprocess_audio(audio, sr):
    # Step 1: Common channel normalization
    # Convert to mono if stereo
    if len(audio.shape) > 1:
        audio = librosa.to_mono(audio)
    
    # Resample to 16kHz
    if sr != 16000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
        sr = 16000
    
    audio = amplify_audio(audio, sr, target_dB= 6000)
    audio = bandpass_filter(audio, sr, lowcut=100, highcut=3400)
    #audio = normalize_loudness(audio, sr)
    audio = denoise_audio(audio, sr)
    
    speech_timestamps = get_vad_segments(audio, sr)
    vad_stats = compute_vad_stats(speech_timestamps, len(audio)/sr, sr)
    
    return audio, sr, vad_stats

def process_participant_audio(audio_path):
    # Load and preprocess single audio file
    audio, sr = librosa.load(audio_path, sr=None)
    preprocessed_audio, sr, vad_stats = preprocess_audio(audio, sr)
    
    return preprocessed_audio, sr, vad_stats

In [76]:
def plot_norm(wave_dict):
    
    n_files = len(wave_dict)
    fig, axes = plt.subplots(n_files, 2, figsize=(12, 3*n_files))
    
    for idx, (sub, data) in enumerate(wave_dict.items()):
        
        audio = data['audio']
        sr = data['sr']
        dur = data['duration']
        
        # Resample to 16kHz
        if sr != 16000:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
            sr = 16000
        
        # Bandpass filter
        
        
        audio = bandpass_filter(audio, sr, lowcut=1, highcut=3400)
        aud_nor = audio
        #aud_nor = normalize_loudness(audio=audio, sr=sr)
        
        aud_den = denoise_audio(audio=aud_nor, sr=sr)
        
        speech_timestamps = get_vad_segments(audio=aud_den, sr=sr)
        vad_stats = compute_vad_stats(speech_timestamps=speech_timestamps, total_duration=len(aud_den)/sr, sr=sr)
        
        wave_dict[sub]['preprocessed'] = aud_den
        wave_dict[sub]['vad_stats'] = vad_stats
        
        # Plot original
        axes[idx, 0].plot(audio)
        axes[idx, 0].set_title(f'Original - {os.path.basename(sub)}')
        
        # Plot preprocessed
        axes[idx, 1].plot(aud_den)
        axes[idx, 1].set_title(f'Preprocessed - {os.path.basename(sub)}')
    
    plt.tight_layout()
    plt.show()
    


In [ ]:
  
from IPython.display import Audio

path = os.getcwd()
parent_dir = os.path.dirname(path)
wave_dict = load_sound(parent_dir)
for idx, (sub, data) in enumerate(wave_dict.items()):
    
    audio = data['audio']
    sr = data['sr']
    dur = data['duration']
    
    preprocessed_audio, sr_new, vad_stats = preprocess_audio(audio, sr)
    
    display(Audio(preprocessed_audio, rate=sr_new))
    plt.figure()
    plt.axvspan(100, 3400, color = 'yellowgreen', alpha = .2)
    spectral_analysis(audio, sr)


In [25]:
path = os.getcwd()
parent_dir = os.path.dirname(path)
wave_dict = load_sound(parent_dir)

#plot_norm(wave_dict)


In [3]:
import os
os.getcwd()

'/Users/minhphan/Documents/DementiaSpeak/codes'